# LUNA CLOUD — Colab Runtime

This notebook connects a Colab GPU runtime outward to your LUNA CLOUD backend.

> **Important: Run all cells FROM TOP TO BOTTOM. Do not skip cells.**

---

## 1️⃣ Set Render Environment

Go to **Render Dashboard → Settings → Environment** and confirm:

```
COMPUTE_PROVIDER=colab
STREAMING_PROVIDER=vnc
```

Then click **Restart service** and wait 3 minutes for the new instance to start.

⚠️ **Without this step, the agent cannot connect.**

## 2️⃣ Set Colab Authentication Secret

Colab sidebar → 🔐 Secrets → Add new secret:

- **Name:** `LUNA_RUNTIME_AUTH`
- **Value:** Paste the value from **Render Dashboard → Settings → Environment → `RUNTIME_AUTH_SECRET`**

⚠️ **This secret must match exactly.** After saving, re-run this cell.

In [ ]:
import os
import sys

# ============================================================
# CONFIGURATION
# ============================================================

LUNA_BACKEND_WS = os.environ.get("LUNA_BACKEND_WS", "wss://kyro-cloud-3fp0.onrender.com/agent")

# Runtime auth secret — must match Render's RUNTIME_AUTH_SECRET
try:
    from google.colab import userdata
    _secret = userdata.get("LUNA_RUNTIME_AUTH")
except Exception:
    _secret = None

RUNTIME_AUTH_SECRET = os.environ.get("RUNTIME_AUTH_SECRET", _secret or "f5bc62648888d9ae066231b0535eb0643fdce19391581df541b19562c9c2a796")
REPO = os.environ.get("LUNA_REPO", "https://github.com/harshpreetsaini/kyo-cloud.git")

os.environ["LUNA_BACKEND_WS"] = LUNA_BACKEND_WS
os.environ["RUNTIME_AUTH_SECRET"] = RUNTIME_AUTH_SECRET

# ============================================================
# DIAGNOSTICS
# ============================================================

print("=== CONFIGURATION CHECK ===")
print("Backend WS:       ", LUNA_BACKEND_WS)
print("Auth secret:      ", "LOADED ✅" if (_secret or RUNTIME_AUTH_SECRET != "runtime-change-me") else "NOT SET ❌")
print("Auth value:       ", repr(RUNTIME_AUTH_SECRET[:8]) + "..." if RUNTIME_AUTH_SECRET != "runtime-change-me" else "UNKNOWN")

if not (_secret or RUNTIME_AUTH_SECRET != "runtime-change-me"):
    print("\n" + "="*50)
    print("❌ FIX THIS FIRST:")
    print("="*50)
    print("1. Click the 🔐 Secrets icon on the LEFT sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: LUNA_RUNTIME_AUTH")
    print("4. Value: (paste Render RUNTIME_AUTH_SECRET)")
    print("5. Re-run this cell after saving the secret.")


## 3️⃣ Expand All Cells & Begin Bootstrap

⚠️ **If you see only logs and no input fields:**

- Click the **▼ chevron** on the left of each cell number to expand
- Press **Shift+Enter** on any cell to expand all and focus the next input
- The bootstrap cell may take 2-5 minutes. Be patient.

---

In [ ]:
import subprocess, time, os
from IPython.display import display, Javascript

def scroll_to_bottom():
    display(Javascript(
    "var element = window.parent.document.querySelector('.jp-OutputArea-output');"
    "if (element) { element.scrollTop = element.scrollHeight; }"
    ))

REPO_DIR = "/content/luna-cloud"
BOOTSTRAP = os.path.join(REPO_DIR, "runtime-agent", "bootstrap", "bootstrap.sh")

def ensure_repo():
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        pull = subprocess.run("cd %s && git pull --ff-only" % REPO_DIR, shell=True, capture_output=True, text=True)
        print("Pull:", (pull.stdout or pull.stderr).strip()[:300])
    else:
        subprocess.run("rm -rf %s" % REPO_DIR, shell=True)
        clone = subprocess.run("git clone https://github.com/harshpreetsaini/kyro-cloud.git %s" % REPO_DIR, shell=True, capture_output=True, text=True)
        print("Clone:", (clone.stdout or clone.stderr).strip()[:300])
    # Fallback: if the checkout is still incomplete, force a fresh clone.
    if not os.path.isfile(BOOTSTRAP):
        print("Checkout incomplete, forcing fresh clone...")
        subprocess.run("rm -rf %s" % REPO_DIR, shell=True)
        clone = subprocess.run("git clone https://github.com/harshpreetsaini/kyro-cloud.git %s" % REPO_DIR, shell=True, capture_output=True, text=True)
        print("Clone:", (clone.stdout or clone.stderr).strip()[:300])

ensure_repo()

if not os.path.isfile(BOOTSTRAP):
    print("ERROR: bootstrap script not found at", BOOTSTRAP)
    print("Repo contents:", subprocess.run("ls -la %s" % REPO_DIR, shell=True, capture_output=True, text=True).stdout)
    raise SystemExit("Bootstrap aborted: repo checkout incomplete.")

print("Running bootstrap — this installs xfce4, tigervnc, dbus-x11, openbox")
print("It will also START the agent with Xvfb virtual display.")
print()

# Use the absolute bootstrap path so it never depends on the current directory.
proc = subprocess.Popen("bash %s" % BOOTSTRAP, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

while True:
    output = proc.stdout.readline()
    if output == '' and proc.poll() is not None:
        break
    if output:
        print(output.rstrip())

proc.wait()
rc = proc.returncode

scroll_to_bottom()
if rc != 0:
    print("\n❌ BOOTSTRAP EXIT CODE: %s" % rc)
    print("Bootstrap failed! Check errors above.")
else:
    print("\n✅ Bootstrap completed.")
    scroll_to_bottom()

time.sleep(3)
print("\n=== AGENT STATUS ===")
r2 = subprocess.run("ps aux | grep '[p]ython3 main.py'", shell=True, capture_output=True, text=True)
print(r2.stdout or "Agent not running!")
scroll_to_bottom()
print()
if r2.stdout:
    print("🟢 Agent is RUNNING")
    print("👉 Next: Go to Vercel, hard-refresh (Ctrl+Shift+R), click Start Session")
    scroll_to_bottom()
else:
    print("🔴 Agent NOT running. Check errors above.")
    scroll_to_bottom()
